In [1]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())


2.10.0+cpu
None
False


In [29]:
import labelme2coco


labelme_train = r"D:\COS40007\Week_05\Data\Task_02\train"
json_train = r"D:\COS40007\Week_05\Data\Task_02\train_coco"

labelme_test = r"D:\COS40007\Week_05\Data\Task_02\test"
json_test = r"D:\COS40007\Week_05\Data\Task_02\test_coco"


def convert_json_to_coco(json_path, export_path):
    """
    Convert JSON annotations to COCO format.
    
    Args:
        json_path (str): Path to the input JSON file.
        export_path (str): Path to save the output COCO JSON file.
    """
    # Convert JSON to COCO format
    labelme2coco.convert(json_path, export_path)
    print(f"Converted {json_path} to COCO format and saved to {export_path}")

# Convert training annotations
convert_json_to_coco(labelme_train, json_train)
# Convert testing annotations
convert_json_to_coco(labelme_test, json_test)

There are 136 listed files in folder train.


Converting labelme annotations to COCO format: 100%|██████████| 136/136 [00:00<00:00, 252.88it/s]
03/22/2026 16:19:28 - INFO - labelme2coco -   Converted annotations in COCO format is exported to D:\COS40007\Week_05\Data\Task_02\train_coco\dataset.json


Converted D:\COS40007\Week_05\Data\Task_02\train to COCO format and saved to D:\COS40007\Week_05\Data\Task_02\train_coco
There are 3 listed files in folder test.


Converting labelme annotations to COCO format: 100%|██████████| 3/3 [00:00<00:00, 270.56it/s]
03/22/2026 16:19:28 - INFO - labelme2coco -   Converted annotations in COCO format is exported to D:\COS40007\Week_05\Data\Task_02\test_coco\dataset.json


Converted D:\COS40007\Week_05\Data\Task_02\test to COCO format and saved to D:\COS40007\Week_05\Data\Task_02\test_coco


In [ ]:
import os
import torch
import torchvision
import numpy as np
import cv2
from torch.utils.data import Dataset, DataLoader
import json
import torchvision.transforms as T

##################################
# SECTION 1: Data Loading
##################################
print("SECTION 1: Data Loading...")


train_json = os.path.join(json_train, "dataset.json")
train_images = labelme_train

test_json = os.path.join(json_test, "dataset.json")
test_images = labelme_test

print(f"Train JSON: {train_json}")
print(f"Train Images Dir: {train_images}")
print(f"Test JSON: {test_json}")
print(f"Test Images Dir: {test_images}")

# Custom Dataset for COCO Annotations
class LogsDataset(Dataset):
    def __init__(self, coco_json, images_dir, transforms=None):
        self.images_dir = images_dir
        self.transforms = transforms
        # Load COCO-like JSON
        with open(coco_json, 'r') as f:
            data = json.load(f)
        self.all_images = data['images']
        self.annotations = data['annotations']
        self.categories = data['categories']

        # Filter images - keep only those that exist
        valid_images = []
        for img_info in self.all_images:
            file_name = os.path.basename(img_info['file_name'])
            img_path = os.path.join(self.images_dir, file_name)
            if os.path.exists(img_path):
                valid_images.append(img_info)
        
        self.images = valid_images
        print(f"Filtered dataset: {len(valid_images)} valid images out of {len(self.all_images)}")

        # Build index of image_id -> list of annotation indexes
        self.img_id_to_ann = {}
        for i, ann in enumerate(self.annotations):
            img_id = ann['image_id']
            if img_id not in self.img_id_to_ann:
                self.img_id_to_ann[img_id] = []
            self.img_id_to_ann[img_id].append(i)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_info = self.images[idx]
        img_id = img_info['id']
        file_name = img_info['file_name']
        
        # Extract just the filename if full path is stored
        file_name = os.path.basename(file_name)
        img_path = os.path.join(self.images_dir, file_name)
        
        # Load image
        img = cv2.imread(img_path)
        if img is None:
            raise FileNotFoundError(f"Cannot load image at path: {img_path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # Convert to RGB
        
        # Gather annotations
        ann_ids = self.img_id_to_ann.get(img_id, [])
        boxes = []
        labels = []
        
        for ann_id in ann_ids:
            ann = self.annotations[ann_id]
            # Single class "log" => label=1
            labels.append(1)
            # Bbox in COCO is [x, y, width, height]
            x, y, w, h = ann['bbox']
            boxes.append([x, y, x + w, y + h])

        # Convert to torch tensors
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        
        masks = torch.zeros((len(boxes), img.shape[0], img.shape[1]), dtype=torch.uint8)

        target = {
            "boxes": boxes,
            "labels": labels,
            "masks": masks
        }

        if self.transforms:
            img = self.transforms(img)

        return img, target


SECTION 1: Data Loading...
Train JSON: D:\COS40007\Week_05\Data\Task_02\train_coco\dataset.json
Train Images Dir: D:\COS40007\Week_05\Data\Task_02\train
Test JSON: D:\COS40007\Week_05\Data\Task_02\test_coco\dataset.json
Test Images Dir: D:\COS40007\Week_05\Data\Task_02\test


In [40]:

# Transforms
transform = T.Compose([
    T.ToTensor(),
])

# Create datasets
dataset_train = LogsDataset(train_json, train_images, transforms=transform)
dataset_test  = LogsDataset(test_json, test_images, transforms=transform)

print(f"Number of training images: {len(dataset_train)}")
print(f"Number of testing images: {len(dataset_test)}")

# Create dataloaders
def collate_fn(batch):
    return tuple(zip(*batch))

train_loader = DataLoader(
    dataset_train,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    dataset_test,
    batch_size=2,
    shuffle=False,
    collate_fn=collate_fn
)

print("Data Loading Complete.\n")

Filtered dataset: 27 valid images out of 136
Filtered dataset: 1 valid images out of 3
Number of training images: 27
Number of testing images: 1
Data Loading Complete.



In [41]:
##################################
# SECTION 2: Model Setup
##################################
print("SECTION 2: Model Setup...")

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"Device being used: {device}")

# Load pre-trained PyTorch Mask R-CNN
model = torchvision.models.detection.maskrcnn_resnet50_fpn(pretrained=True)
print("Loaded pre-trained Mask R-CNN.")

# Replace classifier head (box predictor)
num_classes = 2  # 1 (log) + background
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(
    in_features, num_classes
)

# Replace mask predictor
in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
hidden_layer = 256
model.roi_heads.mask_predictor = torchvision.models.detection.mask_rcnn.MaskRCNNPredictor(
    in_features_mask, hidden_layer, num_classes
)

model.to(device)
print("Model moved to device.")
print("Model Setup Complete.\n")


SECTION 2: Model Setup...
Device being used: cpu
Loaded pre-trained Mask R-CNN.
Model moved to device.
Model Setup Complete.



In [42]:
##################################
# SECTION 3: Training & Testing
##################################
print("SECTION 3: Training & Testing...")

# Optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)

num_epochs = 10
print(f"Starting training for {num_epochs} epochs...")

model.train()
for epoch in range(num_epochs):
    print(f"Starting Epoch {epoch+1}...")
    total_loss = 0.0
    
    for step, (imgs, targets) in enumerate(train_loader):
        imgs = [img.to(device) for img in imgs]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(imgs, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        total_loss += losses.item()
        if (step + 1) % 10 == 0:
            print(f"  [Step {step+1}] Loss: {losses.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}] - Average Loss: {avg_loss:.4f}")

print("Training completed.\n")

SECTION 3: Training & Testing...
Starting training for 10 epochs...
Starting Epoch 1...
  [Step 10] Loss: 1.2130
Epoch [1/10] - Average Loss: 1.7862
Starting Epoch 2...
  [Step 10] Loss: 0.3582
Epoch [2/10] - Average Loss: 0.4675
Starting Epoch 3...
  [Step 10] Loss: 0.2438
Epoch [3/10] - Average Loss: 0.2419
Starting Epoch 4...
  [Step 10] Loss: 0.1712
Epoch [4/10] - Average Loss: 0.1905
Starting Epoch 5...
  [Step 10] Loss: 0.1495
Epoch [5/10] - Average Loss: 0.1658
Starting Epoch 6...
  [Step 10] Loss: 0.1223
Epoch [6/10] - Average Loss: 0.1419
Starting Epoch 7...
  [Step 10] Loss: 0.1167
Epoch [7/10] - Average Loss: 0.1215
Starting Epoch 8...
  [Step 10] Loss: 0.1007
Epoch [8/10] - Average Loss: 0.1124
Starting Epoch 9...
  [Step 10] Loss: 0.1008
Epoch [9/10] - Average Loss: 0.1051
Starting Epoch 10...
  [Step 10] Loss: 0.1130
Epoch [10/10] - Average Loss: 0.1098
Training completed.



In [43]:
##################################
# Debugging: Check File Paths
##################################
print("Checking available images in train directory...")

# List actual files in train directory
train_files = os.listdir(train_images)
print(f"Total files in train directory: {len(train_files)}")
print(f"Sample files: {train_files[:5]}")

# Load dataset.json and check paths
with open(train_json, 'r') as f:
    data = json.load(f)
    
print(f"\nTotal images in dataset.json: {len(data['images'])}")
print(f"Sample paths from dataset.json:")
for i, img_info in enumerate(data['images'][:5]):
    print(f"  - {img_info['file_name']}")

# Check if paths match
missing_count = 0
for img_info in data['images']:
    img_path = os.path.join(train_images, img_info['file_name'])
    if not os.path.exists(img_path):
        missing_count += 1
        if missing_count <= 5:
            print(f"\nMissing: {img_path}")

print(f"\nTotal missing files: {missing_count}/{len(data['images'])}")


Checking available images in train directory...
Total files in train directory: 258
Sample files: ['00305-ZED-Right-1622129312.json', '00308-ZED-Right-1622129468.png', '00309-ZED-Right-1622129510.json', '00310-ZED-Left-1622129553.png', '00311-ZED-Right-1622129599.png']

Total images in dataset.json: 136
Sample paths from dataset.json:
  - D:\COS40007\Week_05\Data\Task_02\train\00305-ZED-Right-1622129312.png
  - D:\COS40007\Week_05\Data\Task_02\train\00309-ZED-Right-1622129510.png
  - D:\COS40007\Week_05\Data\Task_02\train\00314-ZED-Left-1622129763.png
  - D:\COS40007\Week_05\Data\Task_02\train\00316-ZED-Right-1622129908.png
  - D:\COS40007\Week_05\Data\Task_02\train\00317-ZED-Right-1622130197.png

Missing: D:\COS40007\Week_05\Data\Task_02\train\00305-ZED-Right-1622129312.png

Missing: D:\COS40007\Week_05\Data\Task_02\train\00309-ZED-Right-1622129510.png

Missing: D:\COS40007\Week_05\Data\Task_02\train\00314-ZED-Left-1622129763.png

Missing: D:\COS40007\Week_05\Data\Task_02\train\00316-

In [44]:
print("Starting Evaluation / Testing")
model.eval()
log_counts = {}

with torch.no_grad():
    for i, (imgs, targets) in enumerate(test_loader):
        imgs = [img.to(device) for img in imgs]
        outputs = model(imgs)

        for img_i, output in enumerate(outputs):
            boxes = output["boxes"].cpu().numpy()
            scores = output["scores"].cpu().numpy()
            threshold = 0.5
            keep = scores >= threshold
            count_logs = sum(keep)
            log_counts[f"batch{i}_img{img_i}"] = count_logs

print("Testing completed.")
print("Log Counts:", log_counts)

print("\nSECTION 3: Training & Testing Complete.")

Starting Evaluation / Testing
Testing completed.
Log Counts: {'batch0_img0': np.int64(12)}

SECTION 3: Training & Testing Complete.


In [45]:
torch.save(model, 'maskrcnn_model_full.pth')
print("Entire model saved as 'maskrcnn_model_full.pth'")

Entire model saved as 'maskrcnn_model_full.pth'


In [46]:
import cv2
import numpy as np

def draw_predictions(image, boxes, scores, threshold=0.5):
    
    image = np.ascontiguousarray(image)
    for box, score in zip(boxes, scores):
        if score >= threshold:
            x1, y1, x2, y2 = map(int, box)
            cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(image, f"log: {score:.2f}", (x1, y1-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    return image


with torch.no_grad():
    for i, (imgs, targets) in enumerate(test_loader):
        imgs = list(img.to(device) for img in imgs)
        outputs = model(imgs)
        
        for img_i, output in enumerate(outputs):
            # Convert tensor image back to NumPy (after moving to CPU)
            np_img = imgs[img_i].permute(1, 2, 0).cpu().numpy()
            np_img = (np_img * 255).astype(np.uint8)
            
            boxes = output["boxes"].cpu().numpy()
            scores = output["scores"].cpu().numpy()
            
            
            # Draw predictions and capture the modified image
            np_img = draw_predictions(np_img, boxes, scores, threshold=0.5)
            
            # Save result; converting color from RGB to BGR for OpenCV
            cv2.imwrite(f"Data/Task_02/LogOutput/detected_batch{i}_img{img_i}.jpg", cv2.cvtColor(np_img, cv2.COLOR_RGB2BGR))
